# 组成空间分析

以元素分数和化学计量描述符分析训练集与生成结构的组成空间。
运行前设置数据目录和输出目录。


In [ ]:
import csv
import os
import glob
import warnings
warnings.filterwarnings("ignore")

from pymatgen.core import Structure, Composition, Element
from matminer.featurizers.composition.element import (
    ElementFraction,
    Stoichiometry,
)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

In [ ]:
def check_formula(
        input_formula: str,
):
    try:
        comp = Composition(input_formula)
        for elem in comp.elements:
            assert isinstance(elem, Element)
        return True
    except:
        return False
    
    
def formula_to_composite_features(
        input_formula: str,
        has_elem_frac: bool = True,
        has_stoic: bool = True,
):
    assert check_formula(input_formula), f"{input_formula} is invalid"
    comp = Composition(input_formula)
    descriptor = {}
    if has_elem_frac:
        elem_frac_vct = ElementFraction().featurize(comp)
        descriptor.update({"elem_frac": np.array(elem_frac_vct)})
    if has_stoic:
        stoic_vct = Stoichiometry().featurize(comp)
        descriptor.update({"stoic": np.array(stoic_vct)})
    return descriptor


def cif_to_composite_features(
        input_cif_file: str,
        has_elem_frac: bool = True,
        has_stoic: bool = True,
):
    fml = Structure.from_file(input_cif_file).composition.reduced_formula
    assert check_formula(fml), f"{fml} is invalid"
    descriptor = formula_to_composite_features(
        input_formula=fml,
        has_elem_frac=has_elem_frac,
        has_stoic=has_stoic,
    )
    return descriptor

In [ ]:
sources = {
    "dataset": {"dataset_name": "5sets_data", "color": (237, 125, 49), "label": "MP Dataset"},
    "bcn_40": {"dataset_name": "bcn_40_gens", "color": (164, 192, 227), "label": "bcn_40"},
    "bp_40": {"dataset_name": "bp_40_gens", "color": (122, 169, 218), "label": "bp_40"},
    "csi_40": {"dataset_name": "csi_40_gens", "color": (86, 148, 203), "label": "csi_40"},
    "np_40": {"dataset_name": "np_40_gens", "color": (76, 132, 182), "label": "np_40"},
    "sps_30": {"dataset_name": "sps_30_gens", "color": (65, 113, 156), "label": "sps_30"},
}
work_dir = "C:/Users/13701/Desktop/gm_cifs/cdvae/v32/to_plot/"

In [ ]:
filenames_all, elem_frac_feats_all, stoic_feats_all, labels_all = [], [], [], []

import time
start = time.time()
for source_name, source_data in sources.items():
    input_cif_files = glob.glob(os.path.join(work_dir, source_data["dataset_name"], "*.cif"))
    for input_cif_file in input_cif_files:
        input_cif_filename = os.path.basename(input_cif_file).split('.')[0]
        descriptor = cif_to_composite_features(
            input_cif_file,
            has_elem_frac=True,
            has_stoic=True,
        )
        filenames_all.append(input_cif_filename)
        elem_frac_feats_all.append(descriptor["elem_frac"])
        stoic_feats_all.append(descriptor["stoic"])
        labels_all.append(source_name)
        if len(labels_all) % 1000 == 0:
            time1 = time.time()
            print(f"Processed {len(labels_all)} files, took {time1 - start:.2f} seconds")
            start = time.time()
print(len(filenames_all))

In [ ]:
filenames_all_in = np.array(filenames_all)
descriptors_in = np.concatenate((elem_frac_feats_all, stoic_feats_all), axis=1)
# descriptors_in = np.array(elem_frac_feats_all)
# descriptors_in = np.array(stoic_feats_all)
print(filenames_all_in.shape)
print(descriptors_in.shape)

In [ ]:
tsne = TSNE(n_components=2, random_state=42, learning_rate=200.0, init="pca")
tsne_results = tsne.fit_transform(descriptors_in)
print(tsne_results.shape)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
max_value = 95.
scaler = MinMaxScaler(feature_range=(-max_value, max_value))
tsne_results_scaled = scaler.fit_transform(tsne_results)

In [ ]:
import random
random.seed(39)
plt.rcParams["font.family"] = "Calibri"
plt.figure(figsize=(16, 16))
tsne_results_in = tsne_results_scaled
all_indices = []
for source_name, source_data in sources.items():
    # if source_name in ["dataset"]:
    if source_name:
        indices = [i for i, label in enumerate(labels_all) if label == source_name]
        if source_name == "dataset":
            indices = random.sample(indices, 2000)
            print(indices[-5:])
            all_indices += indices
        else:
            indices = random.sample(indices, 500)
            all_indices += indices
        plt.scatter(
            tsne_results_in[indices, 0],
            tsne_results_in[indices, 1],
            color=[c/255 for c in source_data["color"]],
            # edgecolors="black",
            # linewidths=5,
            # facecolors="none",
            s=150,
            label=source_data["label"],
            alpha=0.8,  # transparency
        )

# plt.title("t-SNE Visualization", fontsize=32)
plt.xlim(-100, 100)
plt.ylim(-100, 100)
plt.xlabel(r"t-SNE$_1$", fontsize=40)
plt.ylabel(r"t-SNE$_2$", fontsize=40)
# plt.tick_params(axis="x", which='major', labelsize=32, width=1)
plt.xticks([-100, -75, -50, -25, 0, 25, 50, 75, 100], fontsize=32)
plt.yticks([-75, -50, -25, 0, 25, 50, 75, 100], fontsize=32)
# plt.legend(fontsize=26, loc="upper right", ncol=2)  # 图例
# plt.grid(True)
plt.show()

In [ ]:
csv_file = "./f4a_plot.csv"
datarows = np.concatenate((filenames_all_in.reshape(-1, 1), tsne_results_scaled), axis=1)
datarows_to_plot = datarows[all_indices, :]
import csv
with open(csv_file, "w", newline="") as c_file:
    writer = csv.writer(c_file)
    writer.writerows(datarows_to_plot)